# AI Programming — Lecture 11
## Lab 2-5: MLP for Inverse Kinematics

ABB IRB 2400 dataset을 이용하여 **Inverse Kinematics regression**을 학습합니다.

```text
End-effector pose
(x, y, z, yaw, pitch, roll)
        ↓
       MLP
        ↓
Joint angles
(q1, q2, q3, q4, q5, q6)
```

### 학습 목표
- Forward Kinematics와 Inverse Kinematics의 입력/출력 차이를 설명할 수 있습니다.
- pose를 입력으로 받아 joint angle을 예측하는 MLP를 구성할 수 있습니다.
- input/output standardization을 train set 기준으로 적용할 수 있습니다.
- 관절별 MAE를 계산하고 prediction을 확인할 수 있습니다.
- model과 scaler를 저장하고 새로운 pose에 재사용할 수 있습니다.

### 중요한 점
Inverse kinematics는 forward kinematics보다 어려울 수 있습니다.

같은 end-effector pose를 만드는 joint configuration이 여러 개 존재할 수 있기 때문에
pose → joint angle mapping이 항상 단순한 one-to-one 관계는 아닙니다.

### Colab 실행 안내
300,000개 sample을 Colab에서 다루기 위해 다음 설정을 사용합니다.

- `batch_size = 512`
- `epochs = 50`
- `EarlyStopping(patience=7)`
- T4 등 Colab GPU 권장

## 1. 라이브러리와 Colab 환경 설정

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 50
PATIENCE = 7

tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow version:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "사용하지 않음 (CPU)")

## 2. Google Drive 마운트와 데이터 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

csv_path = (
    '/content/drive/MyDrive/Colab Notebooks/data/'
    'datasetIRB2400.csv'
)

df = pd.read_csv(csv_path)

print("데이터셋 로딩 완료")
print("Shape:", df.shape)
display(df.head())

## 3. Input과 Target 설정

Inverse kinematics:

```text
Input  : x, y, z, yaw, pitch, roll
Target : q1_out ~ q6_out
```

즉, 원하는 end-effector pose로부터 필요한 joint angle을 예측합니다.

In [ ]:
X = df[
    ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
].values

y = df[
    ['q1_out', 'q2_out', 'q3_out', 'q4_out', 'q5_out', 'q6_out']
].values

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Train / Validation / Test 분할

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=SEED,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 5. Input / Output Standardization

- Input pose의 위치와 자세는 scale이 다릅니다.
- Output joint angle도 각 관절별 분포가 다를 수 있습니다.

따라서 input과 output을 각각 standardize합니다.

Scaler는 train set에서만 `fit`합니다.

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)
y_test_scaled = scaler_y.transform(y_test)

## 6. MLP Model

원본 notebook의 구조를 유지합니다.

```text
6
→ Dense(128, ReLU)
→ Dense(128, ReLU)
→ Dense(64, ReLU)
→ Dense(6)
```

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(6,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(6),
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

model.summary()

## 7. Early Stopping과 Model Training

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1,
)

## 8. Learning Curve 확인

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Inverse Kinematics: Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## 9. Test Prediction과 원래 단위 복원

In [ ]:
test_loss, test_mae = model.evaluate(
    X_test_scaled,
    y_test_scaled,
    verbose=0,
)

print(f"Test Loss (scaled MSE): {test_loss:.4f}")
print(f"Test MAE  (scaled): {test_mae:.4f}")

y_pred_scaled = model.predict(
    X_test_scaled,
    batch_size=BATCH_SIZE,
    verbose=0,
)

y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = y_test

mae_real = mean_absolute_error(y_true, y_pred)
rmse_real = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Overall MAE  : {mae_real:.4f}")
print(f"Overall RMSE : {rmse_real:.4f}")
print(f"R2 Score     : {r2:.4f}")

## 10. 관절별 MAE

전체 MAE만 보면 어떤 joint에서 문제가 큰지 알기 어렵습니다.

각 관절별로 별도의 MAE를 계산합니다.

In [ ]:
joint_labels = ["q1", "q2", "q3", "q4", "q5", "q6"]

for i, label in enumerate(joint_labels):
    mae_i = mean_absolute_error(
        y_true[:, i],
        y_pred[:, i],
    )
    print(f"MAE for {label}: {mae_i:.4f}")

## 11. q1 Prediction 시각화

In [ ]:
n_plot = 300

plt.figure(figsize=(10, 5))
plt.plot(
    y_true[:n_plot, 0],
    label="True q1",
)
plt.plot(
    y_pred[:n_plot, 0],
    label="Predicted q1",
)
plt.xlabel("Test Sample Index")
plt.ylabel("q1")
plt.title("Inverse Kinematics: True vs. Predicted q1")
plt.legend()
plt.grid(True)
plt.show()

## 12. Model과 Scaler 저장

In [ ]:
save_dir = (
    '/content/drive/MyDrive/Colab Notebooks/models'
)
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(
    save_dir,
    'mlp_inverse_kinematics.keras'
)
scaler_X_path = os.path.join(
    save_dir,
    'scaler_X_inverse_kinematics.pkl'
)
scaler_y_path = os.path.join(
    save_dir,
    'scaler_y_inverse_kinematics.pkl'
)

model.save(model_path)
joblib.dump(scaler_X, scaler_X_path)
joblib.dump(scaler_y, scaler_y_path)

print("모델 저장:", model_path)
print("입력 scaler 저장:", scaler_X_path)
print("출력 scaler 저장:", scaler_y_path)

## 13. 저장한 Model로 새로운 Pose 예측

In [ ]:
loaded_model = keras.models.load_model(model_path)
loaded_scaler_X = joblib.load(scaler_X_path)
loaded_scaler_y = joblib.load(scaler_y_path)

sample_pose = X_test[0:1]

sample_pose_scaled = loaded_scaler_X.transform(
    sample_pose
)
sample_q_scaled = loaded_model.predict(
    sample_pose_scaled,
    verbose=0,
)
sample_q = loaded_scaler_y.inverse_transform(
    sample_q_scaled
)

print("입력 말단 위치/자세:")
print(sample_pose)

print("\n예측된 관절각:")
print(sample_q)

print("\n실제 관절각:")
print(y_test[0:1])

## 14. 직접 해보기

1. Forward Kinematics와 Inverse Kinematics의 MAE를 비교하세요.
2. 어떤 joint의 MAE가 가장 큰지 확인하세요.
3. hidden layer를 더 크게 만들면 validation loss가 어떻게 변하는지 확인하세요.
4. `batch_size=256`과 `512`의 학습 시간을 비교하세요.
5. 같은 pose에 여러 joint solution이 가능한 상황이 inverse kinematics 학습에 어떤 영향을 줄지 생각해 보세요.

### 체크포인트

```text
Forward Kinematics: joint angles → pose
Inverse Kinematics: pose → joint angles
```

Inverse Kinematics는 mapping의 ambiguity 때문에 더 어려운 regression 문제가 될 수 있습니다.